<a href="https://colab.research.google.com/github/hayatkhan20/umd-urban-heat-exposure/blob/main/notebooks/04_canopy_height.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U \
    earthengine-api \
    geemap \
    geopandas \
    pyogrio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.5/481.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.
google-genai 2.12.1 requires google-auth[requests]<2.56.0,>=2.48.1, but you have google-auth 2.57.0 which is incompatible.


In [ ]:
import ee
import geemap

ee.Authenticate()

PROJECT_ID = "aa-hayatnust"

ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


In [ ]:
import os

REPO_DIRECTORY = "/content/umd-urban-heat-exposure"

if not os.path.exists(REPO_DIRECTORY):
    !git clone https://github.com/hayatkhan20/umd-urban-heat-exposure.git
else:
    print("Repository already exists.")

%cd /content/umd-urban-heat-exposure

Cloning into 'umd-urban-heat-exposure'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 45 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 3.60 MiB | 4.62 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/umd-urban-heat-exposure


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

BOUNDARY_PATH = "data/umd_boundary_final.geojson"

BUILDINGS_PATH = (
    "data/processed/"
    "umd_buildings_with_heights_final.geojson"
)

umd_boundary = (
    gpd.read_file(BOUNDARY_PATH)
    .to_crs("EPSG:4326")
)

buildings = (
    gpd.read_file(BUILDINGS_PATH)
    .to_crs("EPSG:4326")
)

print("Boundary features:", len(umd_boundary))
print("Buildings:", len(buildings))
print(
    "Missing building heights:",
    buildings["analysis_height_m"].isna().sum()
)
print("Boundary valid:", umd_boundary.geometry.is_valid.all())
print("Buildings valid:", buildings.geometry.is_valid.all())

assert len(buildings) == 3694
assert buildings["analysis_height_m"].isna().sum() == 0

# Convert boundary to Earth Engine
umd_ee = geemap.geopandas_to_ee(
    umd_boundary[["geometry"]]
)

aoi = umd_ee.geometry()

print("UMD datasets loaded successfully.")

Boundary features: 1
Buildings: 3694
Missing building heights: 0
Boundary valid: True
Buildings valid: True
UMD datasets loaded successfully.


In [ ]:
CHM_ASSET = (
    "projects/naip-chm/assets/"
    "conus-structure-model"
)

chm_collection = (
    ee.ImageCollection(CHM_ASSET)
    .filterBounds(aoi)
)

chm_tile_count = chm_collection.size().getInfo()

print("NAIP-CHM tiles intersecting UMD:", chm_tile_count)

first_chm = ee.Image(chm_collection.first())

print(
    "CHM bands:",
    first_chm.bandNames().getInfo()
)

print(
    "CHM properties:",
    first_chm.propertyNames().getInfo()
)

# Raw values are stored as height × 100
chm_raw = chm_collection.mosaic()

chm_m = (
    chm_raw
    .updateMask(chm_raw.neq(65535))
    .divide(100)
    .rename("structure_height_m")
    .clip(aoi)
)

print(
    "Native projection:",
    first_chm.projection().getInfo()
)

NAIP-CHM tiles intersecting UMD: 4
CHM bands: ['B0']
CHM properties: ['source_doqq', 'year', 'system:time_end', 'system:id', 'processing_date', 'units', 'utm_zone', 'acquisition_date', 'doy', 'scale_factor', 'system:time_start', 'failed_chips', 'elapsed_seconds', 'model_name', 'system:footprint', 'processed_chips', 'system:version', 'chip_size', 'chip_overlap', 'quad_id', 'quarter', 'naip_resolution_cm', 'system:index', 'system:bands', 'system:band_names']
Native projection: {'type': 'Projection', 'crs': 'EPSG:26918', 'transform': [0.6, 0, 332076, 0, -0.6, 4318560.3]}


In [ ]:
from datetime import datetime, timezone

NAIP_START = "2020-01-01"
NAIP_END = "2025-01-01"

naip = (
    ee.ImageCollection("USDA/NAIP/DOQQ")
    .filterBounds(aoi)
    .filterDate(NAIP_START, NAIP_END)
)

naip_count = naip.size().getInfo()

print("NAIP images found:", naip_count)

if naip_count > 0:
    earliest_ms = naip.aggregate_min(
        "system:time_start"
    ).getInfo()

    latest_ms = naip.aggregate_max(
        "system:time_start"
    ).getInfo()

    earliest_date = datetime.fromtimestamp(
        earliest_ms / 1000,
        tz=timezone.utc
    ).date()

    latest_date = datetime.fromtimestamp(
        latest_ms / 1000,
        tz=timezone.utc
    ).date()

    print("Earliest NAIP date:", earliest_date)
    print("Latest NAIP date:", latest_date)

    print(
        "NAIP bands:",
        ee.Image(naip.first())
        .bandNames()
        .getInfo()
    )

NAIP images found: 8
Earliest NAIP date: 2021-06-17
Latest NAIP date: 2023-09-01
NAIP bands: ['R', 'G', 'B', 'N']


In [ ]:
def add_ndvi(image):
    ndvi_band = image.normalizedDifference(
        ["N", "R"]
    ).rename("NDVI")

    return image.addBands(ndvi_band)


naip_with_ndvi = naip.map(add_ndvi)

# Select the highest-quality green observation
# available at each location
naip_best = naip_with_ndvi.qualityMosaic(
    "NDVI"
)

naip_ndvi = (
    naip_best
    .select("NDVI")
    .clip(aoi)
)

ndvi_stats = naip_ndvi.reduceRegion(
    reducer=ee.Reducer.percentile(
        [5, 25, 50, 75, 95]
    ),
    geometry=aoi,
    scale=2,
    maxPixels=1_000_000_000,
    tileScale=8
).getInfo()

print("NAIP NDVI statistics:")
print(ndvi_stats)

NAIP NDVI statistics:
{'NDVI_p25': 0.105338898068565, 'NDVI_p5': 0.011754514537654534, 'NDVI_p50': 0.4179231913088589, 'NDVI_p75': 0.574269534093943, 'NDVI_p95': 0.6677794051331782}


In [ ]:
# Only transfer geometry to Earth Engine
building_geometries = buildings[
    ["geometry"]
].copy()

buildings_ee = geemap.geopandas_to_ee(
    building_geometries
)

building_mask = (
    ee.Image(0)
    .byte()
    .paint(
        featureCollection=buildings_ee,
        color=1
    )
    .rename("building_mask")
    .clip(aoi)
)

print(
    "Buildings transferred to Earth Engine:",
    buildings_ee.size().getInfo()
)

Buildings transferred to Earth Engine: 3694


In [ ]:
# Vegetation threshold
VEGETATION_NDVI = 0.20

# Stronger vegetation threshold inside building
# footprints. This preserves tree crowns that
# overhang roofs while removing normal rooftops.
BUILDING_OVERHANG_NDVI = 0.35

# Minimum woody vegetation height
MIN_CANOPY_HEIGHT_M = 2.0

# Remove implausible local artefacts
MAX_CANOPY_HEIGHT_M = 60.0

vegetation_mask = naip_ndvi.gte(
    VEGETATION_NDVI
)

valid_height_mask = (
    chm_m.gte(MIN_CANOPY_HEIGHT_M)
    .And(
        chm_m.lte(MAX_CANOPY_HEIGHT_M)
    )
)

# Outside buildings: NDVI >= 0.20
# Inside buildings: retain only strong vegetation,
# allowing overhanging tree crowns
building_condition = (
    building_mask.eq(0)
    .Or(
        naip_ndvi.gte(
            BUILDING_OVERHANG_NDVI
        )
    )
)

canopy_mask = (
    vegetation_mask
    .And(valid_height_mask)
    .And(building_condition)
)

canopy_height = (
    chm_m
    .updateMask(canopy_mask)
    .rename("canopy_height_m")
    .toFloat()
)

print("Canopy-height image created.")

Canopy-height image created.


In [ ]:
canopy_reducer = (
    ee.Reducer.count()
    .combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.percentile(
            [25, 50, 75, 95]
        ),
        sharedInputs=True
    )
)

canopy_statistics = canopy_height.reduceRegion(
    reducer=canopy_reducer,
    geometry=aoi,
    scale=1,
    maxPixels=1_000_000_000,
    tileScale=8
).getInfo()

canopy_area_m2 = (
    ee.Image.pixelArea()
    .updateMask(canopy_mask)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=1,
        maxPixels=1_000_000_000,
        tileScale=8
    )
    .get("area")
    .getInfo()
)

aoi_area_m2 = aoi.area(
    maxError=1
).getInfo()

canopy_cover_percent = (
    canopy_area_m2
    / aoi_area_m2
    * 100
)

print("--- UMD CANOPY HEIGHT ---")
print("Canopy statistics:", canopy_statistics)
print(
    "Canopy area:",
    round(canopy_area_m2 / 1_000_000, 3),
    "km²"
)
print(
    "AOI area:",
    round(aoi_area_m2 / 1_000_000, 3),
    "km²"
)
print(
    "Canopy cover:",
    round(canopy_cover_percent, 2),
    "%"
)

--- UMD CANOPY HEIGHT ---
Canopy statistics: {'canopy_height_m_count': 5813804, 'canopy_height_m_max': 57.529998779296875, 'canopy_height_m_mean': 13.771403997938265, 'canopy_height_m_min': 2, 'canopy_height_m_p25': 7.1199179417375005, 'canopy_height_m_p50': 13.369929671136147, 'canopy_height_m_p75': 19.619860135670493, 'canopy_height_m_p95': 26.368932486149387}
Canopy area: 4.509 km²
AOI area: 10.937 km²
Canopy cover: 41.22 %


In [ ]:
canopy_map = geemap.Map(
    height=650
)

canopy_map.add_basemap(
    "SATELLITE"
)

canopy_map.centerObject(
    umd_ee,
    14
)

canopy_vis = {
    "min": 2,
    "max": 35,
    "palette": [
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837"
    ]
}

canopy_map.addLayer(
    canopy_height,
    canopy_vis,
    "Vegetation canopy height",
    True
)

canopy_map.addLayer(
    naip_ndvi,
    {
        "min": 0,
        "max": 0.8,
        "palette": [
            "brown",
            "yellow",
            "green"
        ]
    },
    "NAIP NDVI",
    False
)

canopy_map.addLayer(
    chm_m,
    {
        "min": 0,
        "max": 35,
        "palette": [
            "white",
            "orange",
            "red"
        ]
    },
    "Original structure height",
    False
)

canopy_map.addLayer(
    building_mask.selfMask(),
    {"palette": ["red"]},
    "Overture building mask",
    False,
    0.5
)

boundary_style = umd_ee.style(
    color="red",
    fillColor="00000000",
    width=3
)

canopy_map.addLayer(
    boundary_style,
    {},
    "UMD boundary"
)

canopy_map

Map(center=[38.9895987982508, -76.94166548679515], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
# Inside the UMD boundary:
# canopy pixels contain height in metres;
# non-canopy pixels contain zero.

canopy_height_export = (
    canopy_height
    .unmask(0)
    .clip(aoi)
    .rename("canopy_height_m")
    .toFloat()
)

print(
    "Export band:",
    canopy_height_export
    .bandNames()
    .getInfo()
)

Export band: ['canopy_height_m']


In [ ]:
import json
import os

summary = {
    "study_area": "UMD College Park analysis area",
    "dataset": "NAIP-CHM CONUS Structure Model",
    "earth_engine_asset": (
        "projects/naip-chm/assets/"
        "conus-structure-model"
    ),
    "source_imagery_dates": {
        "earliest_available_naip": "2021-06-17",
        "latest_available_naip": "2023-09-01"
    },
    "native_resolution_m": 0.6,
    "planned_export_resolution_m": 1,
    "output_crs": "EPSG:26918",
    "vegetation_ndvi_threshold": 0.20,
    "building_overhang_ndvi_threshold": 0.35,
    "minimum_canopy_height_m": 2.0,
    "maximum_canopy_height_m": 60.0,
    "canopy_area_km2": round(
        canopy_area_m2 / 1_000_000,
        4
    ),
    "aoi_area_km2": round(
        aoi_area_m2 / 1_000_000,
        4
    ),
    "canopy_cover_percent": round(
        canopy_cover_percent,
        2
    ),
    "canopy_statistics": canopy_statistics,
    "method": (
        "NAIP-CHM structure height filtered using "
        "NAIP NDVI and Overture building footprints."
    ),
    "important_note": (
        "This is a structural baseline derived primarily "
        "from 2021-2023 imagery, not a 2026 canopy-height map."
    )
}

summary_path = (
    "data/processed/"
    "umd_canopy_height_summary.json"
)

os.makedirs(
    "data/processed",
    exist_ok=True
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=2
    )

print("Created:", summary_path)

Created: data/processed/umd_canopy_height_summary.json


In [ ]:
height_task = ee.batch.Export.image.toDrive(
    image=canopy_height_export,
    description="umd_canopy_height_naip_chm_1m",
    folder="FortyGuard",
    fileNamePrefix="umd_canopy_height_naip_chm_1m",
    region=aoi,
    crs="EPSG:26918",
    scale=1,
    maxPixels=10_000_000_000,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

height_task.start()

print("Export started.")
print("Task ID:", height_task.id)
print("Status:", height_task.status()["state"])

Export started.
Task ID: QE4JGPF65HRGKECQFHGNUA7Z
Status: READY


In [ ]:
task_status = height_task.status()

print("State:", task_status["state"])

if "error_message" in task_status:
    print(
        "Error:",
        task_status["error_message"]
    )

State: RUNNING


In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

shutil.copy2(
    summary_path,
    "/content/drive/MyDrive/FortyGuard/"
    "umd_canopy_height_summary.json"
)

print("Summary copied to Google Drive.")

Mounted at /content/drive
Summary copied to Google Drive.


In [ ]:
import os
import glob

drive_folder = "/content/drive/MyDrive/FortyGuard"

canopy_files = glob.glob(
    os.path.join(
        drive_folder,
        "umd_canopy_height_naip_chm_1m*.tif"
    )
)

print("Files found:", len(canopy_files))

for file_path in canopy_files:
    print("\nFile:", file_path)
    print(
        "Size:",
        round(os.path.getsize(file_path) / 1_000_000, 2),
        "MB"
    )

Files found: 1

File: /content/drive/MyDrive/FortyGuard/umd_canopy_height_naip_chm_1m.tif
Size: 25.35 MB


In [ ]:
import rasterio
import numpy as np

canopy_file = canopy_files[0]

with rasterio.open(canopy_file) as src:
    canopy_array = src.read(1, masked=True)

    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("Data type:", src.dtypes[0])
    print("NoData:", src.nodata)

valid_values = canopy_array.compressed()
canopy_values = valid_values[
    valid_values > 0
]

print("\n--- EXPORTED RASTER CHECK ---")
print("Canopy pixels:", len(canopy_values))
print("Minimum canopy height:", round(float(canopy_values.min()), 2), "m")
print("Mean canopy height:", round(float(canopy_values.mean()), 2), "m")
print("Median canopy height:", round(float(np.median(canopy_values)), 2), "m")
print("Maximum canopy height:", round(float(canopy_values.max()), 2), "m")

CRS: EPSG:26918
Resolution: (1.0, 1.0)
Width: 4219
Height: 4738
Bands: 1
Data type: float32
NoData: None

--- EXPORTED RASTER CHECK ---
Canopy pixels: 4513362
Minimum canopy height: 2.0 m
Mean canopy height: 13.77 m
Median canopy height: 13.42 m
Maximum canopy height: 57.53 m
